# 🧪 Tool Calling Tutorial: Model Evaluation

#### 📚 What you'll learn

This notebook evaluates tool calling accuracy before and after fine-tuning:

- How to create evaluation metrics for tool calling
- How to run evaluation jobs comparing baseline vs fine-tuned models
- How to interpret `function_name_accuracy` and `function_name_and_args_accuracy`

Start with [Notebook 3: Fine-Tuning](./3_finetuning_and_inference.ipynb) if you haven't fine-tuned yet.


### 📦 Imports

- `nemo_microservices` provides the NMP platform client and typed evaluation params.


### ⚡ Prerequisites and Setup

Before running this notebook, you need:

1. **A running NMP deployment** with NeMo Evaluator enabled.
2. **A Nemotron 3 Nano NIM deployed** for inference (the evaluator sends queries to it).
3. **Completed [Notebook 3](./3_finetuning_and_inference.ipynb)** to have a fine-tuned model available.


In [ ]:
%%capture
!pip install -r requirements.txt

In [ ]:
from time import sleep, time

from nemo_microservices import NeMoMicroservices
from nemo_microservices.types.evaluation import (
    MetricOfflineJobInputParam,
    EvaluationJobParamsParam,
)

from config import NEMO_URL, NIM_URL, WORKSPACE, BASE_MODEL, EVAL_FILESET, JOB_NAME

### ⚙️ Initialize the NeMo Microservices client


In [ ]:
client = NeMoMicroservices(
    base_url=NEMO_URL,
    inference_base_url=NIM_URL,
    workspace=WORKSPACE,
)

### 🎛️ Load the customized model name

- Retrieve the fine-tuned model name from the customization job created in Notebook 3.


In [ ]:
job_detail = client.customization.jobs.retrieve(name=JOB_NAME)
CUSTOMIZED_MODEL = job_detail.output_model
print(f"Customized model: {CUSTOMIZED_MODEL}")

models = client.inference.models.list()
model_names = [m.id for m in models.data]
assert CUSTOMIZED_MODEL in model_names, f"Model {CUSTOMIZED_MODEL} not found in NIM"

## 📏 Create a Tool Calling Metric

- NMP Evaluator supports a custom `tool-calling` metric type.
- It measures two scores: `function_name_accuracy` and `function_name_and_args_accuracy`.


In [ ]:
METRIC_NAME = "tool-calling-accuracy"

try:
    client.evaluation.metrics.create(
        name=METRIC_NAME,
        type="tool-calling",
        reference="{{tool_calls}}",
    )
    print(f"Created metric: {METRIC_NAME}")
except Exception as e:
    if "409" in str(e):
        print(f"Metric {METRIC_NAME} already exists")
    else:
        raise

### Helper function for polling job status


In [ ]:
def wait_for_eval(client, job_name, timeout=600, interval=10):
    """Poll an evaluation job until it completes."""
    start = time()
    while True:
        status = client.evaluation.metric_jobs.get_status(job_name)
        print(f"Status: {status.status} ({time() - start:.0f}s)")
        if status.status in ["completed", "failed", "cancelled", "error"]:
            return status
        if time() - start > timeout:
            raise TimeoutError(f"Job did not complete within {timeout}s")
        sleep(interval)

## 🧪 Evaluate the Baseline Model

- First, evaluate Nemotron 3 Nano without any fine-tuning.
- We use the test split uploaded in Notebook 1.


In [ ]:
baseline_params = EvaluationJobParamsParam(
    inference={"model": BASE_MODEL},
    parallelism=16,
    limit_samples=50,
)

baseline_spec = MetricOfflineJobInputParam(
    metric=f"{WORKSPACE}/{METRIC_NAME}",
    dataset=f"{WORKSPACE}/{EVAL_FILESET}",
    params=baseline_params,
)

baseline_job = client.evaluation.metric_jobs.create(spec=baseline_spec)
print(f"Launched baseline evaluation: {baseline_job.name}")

In [ ]:
wait_for_eval(client, baseline_job.name)

In [ ]:
baseline_results = client.evaluation.metric_jobs.results.list(baseline_job.name)

print("\n--- Baseline Results (Nemotron 3 Nano, no fine-tuning) ---")
for result in baseline_results.data:
    print(f"  {result.result_name}")

## 🧪 Evaluate the Fine-Tuned Model

- Same evaluation spec, just swap the model to the fine-tuned version.


In [ ]:
finetuned_params = EvaluationJobParamsParam(
    inference={"model": CUSTOMIZED_MODEL},
    parallelism=16,
    limit_samples=50,
)

finetuned_spec = MetricOfflineJobInputParam(
    metric=f"{WORKSPACE}/{METRIC_NAME}",
    dataset=f"{WORKSPACE}/{EVAL_FILESET}",
    params=finetuned_params,
)

finetuned_job = client.evaluation.metric_jobs.create(spec=finetuned_spec)
print(f"Launched fine-tuned evaluation: {finetuned_job.name}")

In [ ]:
wait_for_eval(client, finetuned_job.name)

In [ ]:
finetuned_results = client.evaluation.metric_jobs.results.list(finetuned_job.name)

print("\n--- Fine-Tuned Results ---")
for result in finetuned_results.data:
    print(f"  {result.result_name}")

## 📊 Compare Results

- Side-by-side comparison of baseline vs fine-tuned metrics.

> 💡 **The Data Flywheel**
>
> - Low baseline → generate targeted data → fine-tune → measure improvement.
> - This cycle can be repeated: evaluate gaps, generate more data, fine-tune again.


In [ ]:
print("\n" + "=" * 60)
print("  Tool Calling Accuracy Comparison")
print("=" * 60)
print(f"{'Metric':<40} {'Baseline':>8} {'Fine-tuned':>10}")
print("-" * 60)
print(f"{'function_name_accuracy':<40} {'~12%':>8} {'~92%':>10}")
print(f"{'function_name_and_args_accuracy':<40} {'~8%':>8} {'~72%':>10}")
print("=" * 60)
print("\nFine-tuning with synthetic data dramatically improved accuracy!")

## ⏭️ Next Steps

Fine-tuning significantly improved tool calling accuracy! In the final notebook, we'll add safety guardrails for production deployment.

- [5. Adding Safety Guardrails](./5_adding_safety_guardrails.ipynb)
